# Econometrics Lab 6 — Binary Choice Models and Sample Selection

**Sorbonne Data Analytics**

**Question of the day:** who participates in the labour market? $works_i = 1$ if person $i$ works, $0$ otherwise. And what is the wage of those who work?

| Exercise | Slides |
|---|---|
| 1. The linear probability model and its limits | 11–13 |
| 2. The logit by maximum likelihood, by hand | 3–8, 17–18, 20 |
| 3. Logit vs probit | 16, 19, 24 |
| 4. Interpretation: marginal effects and odds ratios | 21–26 |
| 5. Goodness of fit, tests and ROC curve | 9, 27–29 |
| 6. Sample selection: the Heckman two-step estimator | 33–39 |

**Python reminder**

| Code | Meaning |
|---|---|
| `sm.Logit(y, X).fit(disp=0)`, `sm.Probit(y, X).fit(disp=0)` | estimation by maximum likelihood (`disp=0` hides the iterations) |
| `.params`, `.llf`, `.predict(X)` | coefficients, maximized log-likelihood, predicted probabilities $F(x\hat\beta)$ |
| `X @ beta` | the vector of $x_i\beta$ for all individuals |
| `stats.norm.cdf(z)`, `stats.norm.pdf(z)` | $\Phi(z)$ and $\phi(z)$ |
| `(condition).sum()`, `(condition).mean()` | number / share of individuals for which the condition is true |

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score, roc_curve

rng = np.random.default_rng(42)

## The data (simulated, so we know the truth)

The decision to work follows a latent-variable model (slide 15):
$$y^*_i = -1.2 + 0.25\, educ_i - 0.03\, age_i - 0.6\, kids_i + u_{1i}, \qquad works_i = 1 \text{ if } y^*_i > 0$$
with $u_1 \sim N(0, 1)$: the **probit** is the true model. `kids` is the number of children under 6.

The log-wage is
$$lwage_i = 1 + 0.08\, educ_i + 0.01\, age_i + u_{2i}$$
but it is **observed only for the people who work**. $u_1$ and $u_2$ are correlated ($\rho = 0.6$, $\sigma_{12} = 0.3$): people who work for unobserved reasons (motivation…) also earn more. This matters in Exercise 6.

In [ ]:
N = 2000
educ = rng.integers(8, 19, N)  # years of education
age = rng.uniform(20, 60, N)
kids = rng.poisson(0.8, N)  # children under 6

# Correlated errors: V(u1) = 1, V(u2) = 0.25, Cov(u1, u2) = 0.3
u1, u2 = rng.multivariate_normal([0, 0], [[1, 0.3], [0.3, 0.25]], N).T

works = (-1.2 + 0.25 * educ - 0.03 * age - 0.6 * kids + u1 > 0).astype(int)
lwage = np.where(
    works == 1, 1 + 0.08 * educ + 0.01 * age + u2, np.nan
)  # NaN = not observed

df = pd.DataFrame(
    {"works": works, "educ": educ, "age": age, "kids": kids, "lwage": lwage}
)
y = df["works"]
X = sm.add_constant(df[["educ", "age", "kids"]])
print(f"Share of people who work: {y.mean():.3f}")
df.head()

---
## Exercise 1 — The linear probability model (slides 11–13)

1. Estimate $works = b_0 + b_1\, educ + b_2\, age + b_3\, kids + \varepsilon$ by OLS with robust (HC1) standard errors. Interpret $\hat b_3$.
2. How many predicted "probabilities" are below 0 or above 1?
3. Why are robust standard errors compulsory in this model?

In [ ]:
lpm = ...  # OLS of y on X, with .fit(cov_type="HC1")
print(lpm.summary().tables[1])

p_lpm = lpm.fittedvalues
print("Predictions < 0:", ..., "   Predictions > 1:", ...)

**Your answer:**

---
## Exercise 2 — The logit by maximum likelihood, by hand (slides 17–18, 20)

For the logit, $F(z) = \dfrac{1}{1 + e^{-z}}$ and the log-likelihood of the sample is
$$LL(\beta) = \sum_i y_i \log F(x_i\beta) + (1 - y_i) \log\big(1 - F(x_i\beta)\big)$$

1. Complete `neg_loglik(beta)` $= -LL(\beta)$ and minimize it with `minimize` (minimizing $-LL$ = maximizing $LL$).
2. Compare with `sm.Logit`: coefficients and maximized log-likelihood.
3. Check slide 20: with a constant in the model, the average predicted probability equals the observed share of $y = 1$.

In [ ]:
def neg_loglik(beta):
    p = ...  # F(x beta) for every individual
    return ...  # minus the log-likelihood (use np.sum and np.log)


res = minimize(
    neg_loglik, x0=np.zeros(X.shape[1]), method="Nelder-Mead"
)  # numerical maximization
logit = ...  # sm.Logit

print(pd.DataFrame({"by hand": res.x, "statsmodels": logit.params}).round(3))
print(f"LL by hand = {-res.fun:.3f},  LL statsmodels = {logit.llf:.3f}")

# 3. Slide 20
print("Mean predicted probability:", ..., "  Observed share:", ...)

**Your answer:**

---
## Exercise 3 — Logit vs probit (slides 16, 19, 24)

1. Estimate the probit. Compute the ratio of the logit to the probit coefficients and compare it with $\pi / \sqrt{3} \approx 1.81$ and with the rule of thumb 1.6.
2. Plot the logit predicted probabilities against the probit ones. Conclude.

In [ ]:
probit = ...  # sm.Probit
print(
    pd.DataFrame({"logit": logit.params, "probit": probit.params, "ratio": ...}).round(
        3
    )
)

plt.scatter(..., ..., s=5)  # probit probabilities on x, logit probabilities on y
plt.plot([0, 1], [0, 1], "r--")
plt.xlabel("probit probability")
plt.ylabel("logit probability")
plt.show()

**Your answer:**

---
## Exercise 4 — Interpreting the coefficients (slides 21–26)

The coefficients only give the **sign** of the effects. For their size:

**(a) Marginal effects at the mean (probit):** $\dfrac{\partial P(y = 1)}{\partial x_k} = \phi(\bar x \hat\beta)\, \hat\beta_k$. Compute them by hand, check with `probit.get_margeff(at="mean")`, and compare with the LPM coefficients.

**(b) Discrete change for the average person (slide 23):** probability of working of the average person with 0 young children, then with 1. The effect is $\Phi(\bar x_1 \hat\beta) - \Phi(\bar x_0 \hat\beta)$.

**(c) Odds ratios (logit):** $e^{\hat\beta_k}$. Interpret the odds ratios of `kids` and `educ`.

In [ ]:
# (a)
x_bar = X.mean()  # the average person (const = 1)
me = ...  # phi(x_bar beta) * beta
print(pd.DataFrame({"ME by hand": me, "LPM": lpm.params}).drop("const").round(4))
print(probit.get_margeff(at="mean").summary())

# (b)
x0 = x_bar.copy()
x0["kids"] = 0
x1 = x_bar.copy()
x1["kids"] = 1
change = ...
print(f"Effect of one young child for the average person: {change:.3f}")

# (c)
odds_ratios = ...
print(odds_ratios.round(3))

**Your answer:**

---
## Exercise 5 — Goodness of fit, tests and ROC curve (slides 9, 27–29)

**(a) Pseudo-$R^2$:** $R^2 = \dfrac{LL_{fit} - LL_0}{LL_{max} - LL_0}$, with $LL_{max} = 0$ and $LL_0$ the log-likelihood of a logit with only a constant. Compare with `logit.prsquared`.

**(b) Likelihood ratio test** of $H_0: \beta_{kids} = 0$: $LR = 2\,(LL_{unrestricted} - LL_{restricted}) \to \chi^2_1$. Compare with the Wald test, $z^2$ where $z$ is the z-statistic of `kids`.

In [ ]:
# (a)
ll0 = ...  # .llf of a logit of y on the constant only: X[["const"]]
pseudo_r2 = ...
print(f"Pseudo-R2 by hand = {pseudo_r2:.4f},  statsmodels = {logit.prsquared:.4f}")

# (b)
restricted = ...  # logit without kids: X.drop(columns="kids")
LR = ...
print(f"LR = {LR:.1f}, p-value = {stats.chi2.sf(LR, df=1):.2g}")
print(f"Wald = z^2 = {logit.tvalues['kids'] ** 2:.1f}")

**(c) Classification.** Predict $\hat y = 1$ if $\hat p \geq 0.5$. Look at the confusion matrix and compute the **sensitivity** (share of $y = 1$ correctly predicted) and the **specificity** (share of $y = 0$ correctly predicted).

**(d) ROC curve.** The ROC curve plots the sensitivity against $1 -$ specificity for every possible cut-off. What does the AUC (area under the curve) measure?

In [ ]:
p_hat = logit.predict(X)
y_pred = (p_hat >= 0.5).astype(int)
print(pd.crosstab(y, y_pred, rownames=["observed"], colnames=["predicted"]))

sensitivity = ...  # hint: y_pred[y == 1] are the predictions for the people who work
specificity = ...
print(f"Sensitivity = {sensitivity:.3f},  specificity = {specificity:.3f}")

fpr, tpr, _ = roc_curve(y, p_hat)
plt.plot(fpr, tpr, label=f"logit (AUC = {roc_auc_score(y, p_hat):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="random guess (AUC = 0.5)")
plt.xlabel("1 - specificity")
plt.ylabel("sensitivity")
plt.legend()
plt.show()

**Your answer:**

---
## Exercise 6 — Sample selection: the Heckman two-step estimator (slides 33–39)

The wage is observed only for those who work. Because $u_1$ and $u_2$ are correlated (slide 36):
$$E(lwage \mid x,\ works = 1) = x_2 b_2 + \sigma_{12}\, \lambda(x_1 b_1), \qquad \lambda(z) = \frac{\phi(z)}{\Phi(z)}$$
so OLS on the workers suffers from an omitted variable: the inverse Mills ratio $\lambda$.

1. Estimate the wage equation by OLS **on the workers only**. Compare with the true values.
2. Heckman two-step: (i) take $x_1 \hat b_1$ from the probit of Exercise 3, estimated on **everybody**; (ii) compute $\hat\lambda$; (iii) regress `lwage` on a constant, `educ`, `age` and $\hat\lambda$, on the workers.
3. What do the sign and the significance of the coefficient on $\hat\lambda$ tell you?
4. `kids` is in the participation equation but not in the wage equation. Why is this exclusion restriction useful (slide 39)?

In [ ]:
xb = X @ probit.params  # x1 b1 for everybody
df["mills"] = ...  # phi(xb) / Phi(xb)
workers = df[df["works"] == 1]

naive = ...  # OLS of lwage on a constant, educ and age (workers only)
heckman = ...  # same regression + mills

true = pd.Series([1, 0.08, 0.01, 0.3], index=["const", "educ", "age", "mills"])
print(
    pd.DataFrame(
        {"true": true, "OLS workers": naive.params, "Heckman": heckman.params},
        index=true.index,
    ).round(4)
)
print(f"p-value of mills: {heckman.pvalues['mills']:.2g}")

**Your answer:**

---
## Summary

| Model | $P(y = 1 \mid x)$ | How to read it | Fit |
|---|---|---|---|
| LPM (OLS) | $x b$ | $b_k$ = change in probability; can leave $[0, 1]$; always heteroskedastic → robust SE | $R^2$ |
| Logit | $\dfrac{1}{1 + e^{-x\beta}}$ | sign of $\beta_k$, marginal effects, odds ratios $e^{\beta_k}$ | pseudo-$R^2$, ROC / AUC |
| Probit | $\Phi(x\beta)$ | sign of $\beta_k$, marginal effects | pseudo-$R^2$, ROC / AUC |

- Logit and probit give almost the same probabilities; their coefficients differ by a scale factor ($\approx 1.6$–$1.8$).
- Tests: Wald (z-test), likelihood ratio $2(LL_u - LL_r)$, both $\chi^2$.
- Outcome observed only for a selected sample → Heckman: probit, inverse Mills ratio, OLS with $\hat\lambda$; needs an exclusion restriction.